# Data Preprocessing

This notebook prepares the cleaned Kickstarter dataset for leakage-safe baseline modeling.

Primary goals:

- define the settled-campaign modeling population;
- define a launch-time prediction boundary;
- separate identifiers, post-launch variables, and feature-engineering source columns;
- split the data before fitting learned preprocessing;
- one-hot encode baseline categorical features;
- standardize baseline numerical features;
- validate the transformed train/test matrices;
- save reproducible train/test partitions and preprocessing artifacts.

This notebook intentionally does **not** create derived features such as `state`,
`tier_step`, text features, date features, interactions, or log-transformed features.
Those belong in `04_feature_engineering.ipynb`.

## Workflow Note

The project documentation originally places train/test splitting in the modeling
notebook. However, encoders and scalers learn parameters from data.

To prevent information leakage, the official train/test split is created **before**
fitting preprocessing in this notebook.

`05_modeling.ipynb` should therefore reuse the saved partitions produced here rather
than create a new split.

In [1]:
import sys
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

## Load Cleaned Data

Preprocessing starts from the deterministic cleaned dataset produced by
`02_clean_data.ipynb`.

In [2]:
cleaned_path = (
    PROJECT_ROOT
    / "data"
    / "Processed-Data"
    / "DSI_kickstarterscrape_dataset_cleaned.csv"
)

clean_df = pd.read_csv(cleaned_path)

print("Cleaned shape:", clean_df.shape)
print("Unique project IDs:", clean_df["project id"].nunique())

Cleaned shape: (45815, 17)
Unique project IDs: 45815


In [3]:
assert clean_df.shape[1] == 17
assert clean_df["project id"].duplicated().sum() == 0
assert clean_df["project id"].notna().all()

## Define the Modeling Population

The current classification target is campaign success.

Only campaigns with settled outcomes are used:

- `successful` → target `1`
- `failed` → target `0`

`live`, `canceled`, and `suspended` projects are excluded from the binary modeling
population because they do not represent the two settled target classes.

In [4]:
settled_df = (
    clean_df[
        clean_df["status"].isin(["successful", "failed"])
    ]
    .copy()
)

settled_df["target"] = (
    settled_df["status"]
    .eq("successful")
    .astype(int)
)

print("Settled campaigns:", len(settled_df))
print()
print(settled_df["status"].value_counts())
print()
print("Success rate:", f"{settled_df['target'].mean():.2%}")

Settled campaigns: 41841

status
successful    22902
failed        18939
Name: count, dtype: int64

Success rate: 54.74%


## Define the Prediction-Time Feature Boundary

The baseline model is treated as a **launch-time** prediction model.

Some source columns are therefore excluded from the baseline feature matrix:

### Identifiers

- `project id`
- `url`

These identify records but should not be treated as ordinary predictive features.

### Post-launch / outcome-related variables

- `pledged`
- `funded percentage`
- `backers`
- `updates`
- `comments`

These may contain information generated after campaign launch and can create
information leakage for a launch-time prediction task.

### Deferred feature-engineering sources

- `name`
- `location`
- `funded date`
- `reward levels`

These contain useful information, but they should first be converted into purposeful
derived features in `04_feature_engineering.ipynb` rather than directly one-hot
encoded as high-cardinality raw strings.

The baseline preprocessing step therefore uses:

**Categorical**
- `category`
- `subcategory`

**Numerical**
- `goal`
- `levels`
- `duration`

In [5]:
IDENTIFIER_COLUMNS = [
    "project id",
    "url",
]

POST_LAUNCH_COLUMNS = [
    "pledged",
    "funded percentage",
    "backers",
    "updates",
    "comments",
]

DEFERRED_FEATURE_SOURCES = [
    "name",
    "location",
    "funded date",
    "reward levels",
]

CATEGORICAL_FEATURES = [
    "category",
    "subcategory",
]

NUMERICAL_FEATURES = [
    "goal",
    "levels",
    "duration",
]

BASE_FEATURES = (
    CATEGORICAL_FEATURES
    + NUMERICAL_FEATURES
)

BASE_FEATURES

['category', 'subcategory', 'goal', 'levels', 'duration']

## Validate Baseline Features

Before splitting or transforming the data, verify that the selected baseline
features are present and inspect their missingness.

The cleaned baseline features are complete, so no statistical imputation is required
for this baseline pipeline.

In [6]:
missing_base_features = (
    settled_df[BASE_FEATURES]
    .isna()
    .sum()
)

missing_base_features

category       0
subcategory    0
goal           0
levels         0
duration       0
dtype: int64

In [7]:
assert (missing_base_features == 0).all()

In [8]:
baseline_schema = pd.DataFrame({
    "dtype": settled_df[BASE_FEATURES].dtypes.astype(str),
    "unique": settled_df[BASE_FEATURES].nunique(),
})

baseline_schema

,dtype,unique
category,object,13
subcategory,object,49
goal,float64,1707
levels,int64,63
duration,float64,5885


## Create the Official Train/Test Split

The split is stratified so the successful/failed class ratio remains approximately
the same in both partitions.

A fixed `random_state=42` makes the split reproducible for the team.

In [9]:
X = settled_df.drop(
    columns=[
        "target",
        "status",
    ]
)

y = settled_df["target"]

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y,
)

print("Training rows:", len(X_train_raw))
print("Test rows:", len(X_test_raw))

print(
    "Training success rate:",
    f"{y_train.mean():.2%}"
)

print(
    "Test success rate:",
    f"{y_test.mean():.2%}"
)

Training rows: 29288
Test rows: 12553
Training success rate: 54.74%
Test success rate: 54.74%


In [10]:
assert set(X_train_raw.index).isdisjoint(
    set(X_test_raw.index)
)

assert len(X_train_raw) + len(X_test_raw) == len(settled_df)

## Preserve Split Membership

Project IDs are saved for the train and test partitions.

`04_feature_engineering.ipynb` can use these IDs to reconstruct the exact same
partitions from the cleaned source dataset before creating derived features. This
prevents accidental re-splitting and keeps the team workflow reproducible.

In [11]:
train_split_ids = pd.DataFrame({
    "project id": X_train_raw["project id"].to_numpy(),
    "target": y_train.to_numpy(),
})

test_split_ids = pd.DataFrame({
    "project id": X_test_raw["project id"].to_numpy(),
    "target": y_test.to_numpy(),
})

print(train_split_ids.shape)
print(test_split_ids.shape)

(29288, 2)
(12553, 2)


## Check for Train/Test Category Differences

`OneHotEncoder(handle_unknown="ignore")` is used so categories that appear only in
the test set will not cause preprocessing to fail.

The audit below records whether any such categories exist in this split.

In [12]:
for col in CATEGORICAL_FEATURES:
    train_values = set(
        X_train_raw[col].dropna().unique()
    )
    test_values = set(
        X_test_raw[col].dropna().unique()
    )

    unseen_in_test = sorted(
        test_values - train_values
    )

    print(
        f"{col}: "
        f"{len(train_values)} train values, "
        f"{len(test_values)} test values, "
        f"{len(unseen_in_test)} unseen test values"
    )

    if unseen_in_test:
        print("  Unseen:", unseen_in_test)

category: 13 train values, 13 test values, 0 unseen test values
subcategory: 49 train values, 49 test values, 0 unseen test values


## Build the Preprocessing Pipeline

Two learned transformations are used:

### Categorical features

`OneHotEncoder` converts `category` and `subcategory` into numeric indicator
features. `handle_unknown="ignore"` prevents unseen test categories from causing
errors.

### Numerical features

`StandardScaler` standardizes `goal`, `levels`, and `duration`.

The scaler learns the training-set mean and standard deviation, then applies the same
parameters to the test set.

No mean/median imputation is performed here because the selected baseline features
contain no missing values.

In [13]:
def make_one_hot_encoder():
    try:
        return OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False,
        )
    except TypeError:
        # Compatibility with older scikit-learn versions.
        return OneHotEncoder(
            handle_unknown="ignore",
            sparse=False,
        )


categorical_pipeline = Pipeline(
    steps=[
        (
            "onehot",
            make_one_hot_encoder(),
        ),
    ]
)

numerical_pipeline = Pipeline(
    steps=[
        (
            "scaler",
            StandardScaler(),
        ),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            categorical_pipeline,
            CATEGORICAL_FEATURES,
        ),
        (
            "numerical",
            numerical_pipeline,
            NUMERICAL_FEATURES,
        ),
    ],
    remainder="drop",
)

## Fit on Training Data Only

This is the key leakage-prevention rule:

- `fit_transform()` is used on the training data;
- `transform()` is used on the test data.

The test set never contributes to learned category mappings, numerical means, or
numerical standard deviations.

In [14]:
X_train_base = X_train_raw[BASE_FEATURES]
X_test_base = X_test_raw[BASE_FEATURES]

X_train_preprocessed_array = (
    preprocessor.fit_transform(
        X_train_base
    )
)

X_test_preprocessed_array = (
    preprocessor.transform(
        X_test_base
    )
)

print(
    "Train transformed shape:",
    X_train_preprocessed_array.shape
)

print(
    "Test transformed shape:",
    X_test_preprocessed_array.shape
)

Train transformed shape: (29288, 65)
Test transformed shape: (12553, 65)


## Restore Feature Names

The transformed arrays are converted back to DataFrames with explicit feature names
so later modeling and interpretation remain readable.

In [15]:
feature_names = (
    preprocessor
    .get_feature_names_out()
)

X_train_preprocessed = pd.DataFrame(
    X_train_preprocessed_array,
    columns=feature_names,
    index=X_train_raw.index,
)

X_test_preprocessed = pd.DataFrame(
    X_test_preprocessed_array,
    columns=feature_names,
    index=X_test_raw.index,
)

X_train_preprocessed.head()

,categorical__category_Art,categorical__category_Comics,categorical__category_Dance,categorical__category_Design,categorical__category_Fashion,categorical__category_Film & Video,categorical__category_Food,categorical__category_Games,categorical__category_Music,categorical__category_Photography,...,categorical__subcategory_Sculpture,categorical__subcategory_Short Film,categorical__subcategory_Technology,categorical__subcategory_Theater,categorical__subcategory_Video Games,categorical__subcategory_Webseries,categorical__subcategory_World Music,numerical__goal,numerical__levels,numerical__duration
24643,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.469408,-0.227560,1.566138
45430,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-0.073600,1.693054,-0.568566
44172,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-0.209984,1.452977,-0.570815
42715,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-0.136741,-1.427943,-0.231152
16777,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-0.174625,0.252594,-0.056822


In [16]:
print(
    "Number of transformed features:",
    len(feature_names)
)

print()
print("First 20 feature names:")
print(feature_names[:20])

Number of transformed features: 65

First 20 feature names:
['categorical__category_Art' 'categorical__category_Comics'
 'categorical__category_Dance' 'categorical__category_Design'
 'categorical__category_Fashion' 'categorical__category_Film & Video'
 'categorical__category_Food' 'categorical__category_Games'
 'categorical__category_Music' 'categorical__category_Photography'
 'categorical__category_Publishing' 'categorical__category_Technology'
 'categorical__category_Theater' 'categorical__subcategory_Animation'
 'categorical__subcategory_Art' 'categorical__subcategory_Art Book'
 'categorical__subcategory_Board & Card Games'
 "categorical__subcategory_Children's Book"
 'categorical__subcategory_Classical Music'
 'categorical__subcategory_Comics']


## Validate Numerical Standardization

`StandardScaler` should produce training-set numerical features with means near `0`
and population standard deviations near `1`.

The test-set values are not expected to have exactly zero mean because the scaler is
not fitted on the test data.

In [17]:
numeric_feature_names = [
    name
    for name in feature_names
    if any(
        name.endswith(f"__{col}")
        or name == col
        for col in NUMERICAL_FEATURES
    )
]

scaled_train_numeric = (
    X_train_preprocessed[
        numeric_feature_names
    ]
)

scaling_check = pd.DataFrame({
    "mean": scaled_train_numeric.mean(),
    "std_ddof0": scaled_train_numeric.std(ddof=0),
})

scaling_check

,mean,std_ddof0
numerical__goal,4.636133e-17,1.0
numerical__levels,-1.722726e-16,1.0
numerical__duration,-4.597373e-17,1.0


In [18]:
assert np.allclose(
    scaling_check["mean"].to_numpy(),
    0,
    atol=1e-10,
)

assert np.allclose(
    scaling_check["std_ddof0"].to_numpy(),
    1,
    atol=1e-10,
)

## Validate the Transformed Matrices

The processed train/test matrices should:

- have the same columns in the same order;
- contain no missing values;
- contain no infinite values;
- preserve the expected train/test row counts.

In [19]:
print(
    "Train missing values:",
    X_train_preprocessed.isna().sum().sum()
)

print(
    "Test missing values:",
    X_test_preprocessed.isna().sum().sum()
)

print(
    "Train infinite values:",
    np.isinf(
        X_train_preprocessed.to_numpy()
    ).sum()
)

print(
    "Test infinite values:",
    np.isinf(
        X_test_preprocessed.to_numpy()
    ).sum()
)

Train missing values: 0
Test missing values: 0
Train infinite values: 0
Test infinite values: 0


In [20]:
assert list(
    X_train_preprocessed.columns
) == list(
    X_test_preprocessed.columns
)

assert X_train_preprocessed.isna().sum().sum() == 0
assert X_test_preprocessed.isna().sum().sum() == 0

assert np.isinf(
    X_train_preprocessed.to_numpy()
).sum() == 0

assert np.isinf(
    X_test_preprocessed.to_numpy()
).sum() == 0

assert len(X_train_preprocessed) == len(y_train)
assert len(X_test_preprocessed) == len(y_test)

## Class-Balance Check

The settled Kickstarter target is relatively balanced compared with many binary
classification problems.

No resampling is applied during preprocessing. Class weighting or resampling can be
evaluated later only if model evaluation shows a need for it.

In [21]:
class_balance = pd.DataFrame({
    "train": y_train.value_counts(normalize=True).sort_index(),
    "test": y_test.value_counts(normalize=True).sort_index(),
})

class_balance.index = [
    "failed",
    "successful",
]

class_balance

,train,test
failed,0.452643,0.452641
successful,0.547357,0.547359


## Save Preprocessing Outputs

The generated files are stored under `data/Processed-Data/`, which is excluded from
Git version control.

Saved artifacts:

- baseline preprocessed training matrix;
- baseline preprocessed test matrix;
- train/test targets;
- train/test project-ID membership;
- fitted baseline preprocessor.

The split-ID files are especially important because feature engineering should reuse
the same partitions rather than create a new random split.

In [22]:
processed_dir = (
    PROJECT_ROOT
    / "data"
    / "Processed-Data"
)

processed_dir.mkdir(
    parents=True,
    exist_ok=True,
)

X_train_path = (
    processed_dir
    / "X_train_base_preprocessed.csv"
)

X_test_path = (
    processed_dir
    / "X_test_base_preprocessed.csv"
)

y_train_path = (
    processed_dir
    / "y_train.csv"
)

y_test_path = (
    processed_dir
    / "y_test.csv"
)

train_ids_path = (
    processed_dir
    / "train_project_ids.csv"
)

test_ids_path = (
    processed_dir
    / "test_project_ids.csv"
)

preprocessor_path = (
    processed_dir
    / "base_preprocessor.joblib"
)

X_train_preprocessed.to_csv(
    X_train_path,
    index=False,
)

X_test_preprocessed.to_csv(
    X_test_path,
    index=False,
)

pd.DataFrame({
    "target": y_train.to_numpy()
}).to_csv(
    y_train_path,
    index=False,
)

pd.DataFrame({
    "target": y_test.to_numpy()
}).to_csv(
    y_test_path,
    index=False,
)

train_split_ids.to_csv(
    train_ids_path,
    index=False,
)

test_split_ids.to_csv(
    test_ids_path,
    index=False,
)

joblib.dump(
    preprocessor,
    preprocessor_path,
)

print("Saved preprocessing outputs.")

Saved preprocessing outputs.


## Reload and Validate Saved Outputs

Reloading the generated files verifies that the preprocessing outputs can be consumed
by later notebooks independently of this notebook's in-memory state.

In [23]:
saved_X_train = pd.read_csv(
    X_train_path
)

saved_X_test = pd.read_csv(
    X_test_path
)

saved_y_train = pd.read_csv(
    y_train_path
)

saved_y_test = pd.read_csv(
    y_test_path
)

saved_train_ids = pd.read_csv(
    train_ids_path
)

saved_test_ids = pd.read_csv(
    test_ids_path
)

loaded_preprocessor = joblib.load(
    preprocessor_path
)

assert saved_X_train.shape == X_train_preprocessed.shape
assert saved_X_test.shape == X_test_preprocessed.shape

assert len(saved_y_train) == len(y_train)
assert len(saved_y_test) == len(y_test)

assert len(saved_train_ids) == len(y_train)
assert len(saved_test_ids) == len(y_test)

assert saved_train_ids["project id"].is_unique
assert saved_test_ids["project id"].is_unique

assert set(
    saved_train_ids["project id"]
).isdisjoint(
    set(saved_test_ids["project id"])
)

print("Saved preprocessing outputs validated.")

Saved preprocessing outputs validated.


# Preprocessing Summary and Handoff

`03_preprocess_data.ipynb` establishes a leakage-safe baseline preprocessing
workflow.

## Completed in this notebook

- restricted the modeling population to `successful` and `failed` campaigns;
- created a binary success target;
- defined a launch-time prediction boundary;
- excluded identifiers from baseline model features;
- excluded post-launch/outcome-related variables from the launch-time baseline;
- deferred high-cardinality raw text/location/date/reward fields to feature
  engineering;
- created one reproducible stratified train/test split;
- verified baseline feature missingness;
- one-hot encoded `category` and `subcategory`;
- standardized `goal`, `levels`, and `duration`;
- fitted all learned preprocessing on training data only;
- transformed the test data using the already-fitted training preprocessor;
- validated feature names, shapes, missing values, infinities, and scaled training
  statistics;
- reviewed class balance without applying unnecessary resampling;
- saved the train/test matrices, targets, split membership, and fitted preprocessor.

## Relationship to earlier machine-learning work

This pipeline preserves the same core principles used in earlier coursework:

- inspect missing values before deciding how to treat them;
- convert categorical variables into numeric representations;
- standardize numerical variables when the model benefits from comparable scales;
- split data before fitting learned preprocessing;
- fit a scaler/encoder using training data and reuse it on test data.

The difference is that preprocessing is now organized into a reusable
`ColumnTransformer`/`Pipeline` rather than being applied manually column by column.

## No statistical imputation was needed

The selected baseline features contain no missing values after deterministic cleaning.

Therefore, mean/median imputation and categorical `Unknown` replacement are not
applied merely for the sake of applying them. If later engineered features contain
missing values, their treatment should be added to the pipeline and fitted using the
training partition only.

## Deferred to feature engineering

`04_feature_engineering.ipynb` should reuse `train_project_ids.csv` and
`test_project_ids.csv` to preserve the same train/test membership while creating
features such as:

- state/location-derived features;
- `tier_step` and other reward-tier summaries;
- date-derived features;
- missingness indicators where meaningful;
- transformed skewed numerical features;
- justified interaction features.

The baseline preprocessed matrices produced here can already support a simple
leakage-safe baseline model, while later engineered features can extend the same
train/test partitions.